In [1]:
# Cài đặt thư viện tính toán metric OCR nếu chưa có
!pip install jiwer -q

import os
import json
import torch
import jiwer
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# =====================================================================
# ⚙️ CONFIGURATION (CẤU HÌNH ĐƯỜNG DẪN)
# =====================================================================
# Model chính lấy từ Kansallisarkisto
MODEL_HF_REPO = "Kansallisarkisto/cyrillic-htr-model"
# Bộ Processor lấy từ model base của Microsoft để tránh lỗi đường dẫn Hub
PROCESSOR_HF_REPO = "microsoft/trocr-base-handwritten" 

IMAGE_DIR = "/kaggle/input/datasets/quii29/rukopys-dataset/train/images"
TEST_JSONL_PATH = "/kaggle/input/datasets/habao2603/metadata-for-paper/test.jsonl" 
REPORT_OUTPUT_PATH = "/kaggle/working/report.json"

# 👇 ĐƯỜNG DẪN XUẤT FILE SUBMISSION CSV MỚI BỔ SUNG
SUBMISSION_OUTPUT_PATH = "/kaggle/working/submission.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HPA_TYPES = {"handwritten", "printed", "annotation"}

print(f"🚀 Khởi tạo Pipeline Đánh Giá OCR & Tạo Submission. Thiết bị sử dụng: {DEVICE.upper()}")

# =====================================================================
# 🤖 STEP 1: TẢI MÔ HÌNH, THÊM KÝ TỰ UKRAINE VÀ KHỞI TẠO BỘ ĐẾM
# =====================================================================
print(f"🔄 Đang nạp TrOCR Processor từ: {PROCESSOR_HF_REPO}")
print(f"🔄 Đang nạp TrOCR Model từ: {MODEL_HF_REPO}...")

try:
    # 1. Tải bộ xử lý ảnh & text
    processor = TrOCRProcessor.from_pretrained(PROCESSOR_HF_REPO)
    
    # 2. Tải trọng số mô hình
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_HF_REPO).to(DEVICE)
    
    print("✨ Thêm các ký tự đặc trưng của tiếng Ukraine vào Vocab...")
    # Danh sách ký tự Ukraine độc nhất (cả viết hoa, viết thường và dấu nháy đơn đặc trưng)
    ukrainian_tokens = ['Ґ', 'ґ', 'Є', 'є', 'І', 'і', 'Ї', 'ї', '’']
    
    # Thêm các ký tự này vào tokenizer
    num_added_tokens = processor.tokenizer.add_tokens(ukrainian_tokens)
    print(f"🔹 Đã thêm thành công {num_added_tokens} token mới vào Tokenizer.")
    
    # 3. Mở rộng kích thước Embedding của DECODER tương ứng với Vocab mới
    if num_added_tokens > 0:
        # 👉 THAY ĐỔI CHÍNH: Gọi vào model.decoder thay vì model
        model.decoder.resize_token_embeddings(len(processor.tokenizer))
        
        # Đồng bộ lại cấu hình hệ thống để tránh lỗi lệch cấu trúc khi lưu/tải model
        model.config.decoder.vocab_size = len(processor.tokenizer)
        model.config.vocab_size = len(processor.tokenizer)
        
        print(f"🔹 Đã cập nhật kích thước Embedding của Decoder thành: {len(processor.tokenizer)}")
        
    model.eval()
    print("✅ Khởi tạo mô hình và mở rộng Vocab thành công!")

except Exception as e:
    print(f"❌ [CRITICAL ERROR] Không thể tải hoặc cấu hình model: {e}")
    raise e

# Cấu trúc lưu trữ ground-truth (gt) và prediction (pred) cho từng loại metric
evaluation_data = {
    "handwritten": {"gt": [], "pred": []},
    "printed": {"gt": [], "pred": []},
    "annotation": {"gt": [], "pred": []},
    "overall": {"gt": [], "pred": []}
}

# Đếm số lượng dòng trong file jsonl để làm thanh tiến trình tqdm
total_lines = 0
if os.path.exists(TEST_JSONL_PATH):
    with open(TEST_JSONL_PATH, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)
else:
    raise FileNotFoundError(f"❌ Không tìm thấy file test.jsonl tại: '{TEST_JSONL_PATH}'")

# =====================================================================
# 🧠 STEP 2: CHẠY INFERENCE TRÊN TẬP TEST VÀ THU THẬP KẾT QUẢ
# =====================================================================
print(f"⚡ Bắt đầu chạy inference trên {total_lines} ảnh từ tập test...")

# 👇 Khởi tạo danh sách lưu trữ bản ghi cho file submission CSV
submission_records = []

with open(TEST_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in tqdm(f, total=total_lines, desc="Evaluating"):
        data = json.loads(line)
        
        # Trích xuất tên file ảnh gốc (đề phòng đường dẫn chứa tiền tố như 'images/')
        img_name = os.path.basename(data['file_name'])
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        if not os.path.exists(img_path):
            # Thử tìm trực tiếp nếu cấu trúc thư mục khác biệt
            img_path = os.path.join(IMAGE_DIR, data['file_name'])
            if not os.path.exists(img_path):
                print(f"Không tìm thấy ảnh: {img_name}")
                continue
                
        try:
            img = Image.open(img_path).convert("RGB")
            w, h = img.size
        except Exception as e:
            print(f"❌ Không thể mở file ảnh {img_name}: {e}")
            continue
            
        regions = data.get('regions', [])
        
        # 👇 Mảng tạm lưu các vùng (gồm cả vùng OCR và vùng loại khác) của ẢNH HIỆN TẠI
        pred_regions = []
        
        for region in regions:
            r_type = region.get('type')
            bbox = region.get('bbox')
            # Mặc định lấy text cũ (giúp giữ nguyên text của các type không thuộc HPA_TYPES như formula, table, image...)
            pred_text = region.get('text', '').strip()
            
            # Chỉ chạy OCR với các type cần đánh giá nằm trong HPA_TYPES
            if r_type in HPA_TYPES:
                gt_text = region.get('text', '').strip()
                
                if not bbox or len(bbox) != 4:
                    continue
                    
                x1, y1, x2, y2 = bbox
                # Clip coordinates chống tràn viền tương tự code gốc của bạn
                x1, x2 = max(0, min(int(x1), w)), max(0, min(int(x2), w))
                y1, y2 = max(0, min(int(y1), h)), max(0, min(int(y2), h))
                
                if x2 <= x1 or y2 <= y1:
                    continue
                    
                # Crop vùng ảnh chứa text
                cropped_img = img.crop((x1, y1, x2, y2))
                
                # Dự đoán chuỗi ký tự bằng TrOCR
                try:
                    pixel_values = processor(images=cropped_img, return_tensors="pt").pixel_values.to(DEVICE)
                    with torch.no_grad():
                        generated_ids = model.generate(pixel_values, max_length=128)
                    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
                except Exception as e:
                    print(f"❌ Lỗi xử lý tại bbox {bbox} của ảnh {img_name}: {e}")
                    pred_text = ""  # Đặt chuỗi rỗng nếu lỗi hoàn toàn
                
                # Lưu kết quả vào danh mục tương ứng phục vụ tính Metric ở STEP 3
                evaluation_data[r_type]["gt"].append(gt_text)
                evaluation_data[r_type]["pred"].append(pred_text)
                
                # Đồng thời lưu vào nhóm tổng thể (overall)
                evaluation_data["overall"]["gt"].append(gt_text)
                evaluation_data["overall"]["pred"].append(pred_text)
            
            # 👇 Thêm vùng (đã cập nhật pred_text mới nếu qua OCR hoặc giữ nguyên text gốc) vào danh sách
            pred_regions.append({
                "bbox": bbox,
                "type": r_type,
                "text": pred_text
            })
            
        # 👇 Lưu bản ghi của ảnh hiện tại vào danh sách dữ liệu submission
        submission_records.append({
            "image": img_name,
            "regions": json.dumps(pred_regions, ensure_ascii=False)
        })

# =====================================================================
# 📊 STEP 3: TÍNH TOÁN CÁC CHỈ SỐ CER, WER, EXACT MATCH (EM)
# =====================================================================
print("\n🔄 Đang tính toán các chỉ số lỗi (Metrics Calculation)...")
def calculate_metrics_robustly(gt_list, pred_list):
    if not gt_list:
        return {"CER": 0.0, "WER": 0.0, "Exact_Match": 0.0, "Total_Samples": 0}
        
    total_char_dist = 0
    total_char_len = 0
    total_word_dist = 0
    total_word_len = 0
    em_count = 0
    
    for gt, pred in zip(gt_list, pred_list):
        if gt == pred:
            em_count += 1
            
        if len(gt) == 0:
            total_char_dist += len(pred)
        else:
            try:
                char_error = jiwer.cer(gt, pred) * len(gt)
                total_char_dist += char_error
            except:
                total_char_dist += len(pred)
            total_char_len += len(gt)
            
        gt_words = gt.split()
        pred_words = pred.split()
        if len(gt_words) == 0:
            total_word_dist += len(pred_words)
        else:
            try:
                word_error = jiwer.wer(gt, pred) * len(gt_words)
                total_word_dist += word_error
            except:
                total_word_dist += len(pred_words)
            total_word_len += len(gt_words)

    cer = total_char_dist / total_char_len if total_char_len > 0 else (0.0 if total_char_dist == 0 else 1.0)
    wer = total_word_dist / total_word_len if total_word_len > 0 else (0.0 if total_word_dist == 0 else 1.0)
    em = em_count / len(gt_list)
    
    return {
        "CER": round(float(cer), 5),
        "WER": round(float(wer), 5),
        "Exact_Match": round(float(em), 5),
        "Total_Samples": len(gt_list)
    }

report_dict = {}
for category, data in evaluation_data.items():
    metrics = calculate_metrics_robustly(data["gt"], data["pred"])
    report_dict[category] = metrics

# =====================================================================
# 💾 STEP 4: XUẤT FILE REPORT.JSON VÀ HIỂN THỊ KẾT QUẢ
# =====================================================================
with open(REPORT_OUTPUT_PATH, 'w', encoding='utf-8') as json_f:
    json.dump(report_dict, json_f, ensure_ascii=False, indent=4)

print("\n" + "="*60)
print("🏁 TIẾN TRÌNH ĐÁNH GIÁ HOÀN TẤT!")
print(f"💾 File báo cáo metric đã được lưu tại: {REPORT_OUTPUT_PATH}")
print("="*60)

df_report = pd.DataFrame(report_dict).T
print(df_report.to_string())

# =====================================================================
# 💾 STEP 5: XUẤT FILE SUBMISSION.CSV THEO ĐÚNG FORMAT
# =====================================================================
print("\n" + "="*60)
print("📦 ĐANG XUẤT FILE SUBMISSION CSV...")

df_submission = pd.DataFrame(submission_records)
df_submission.to_csv(SUBMISSION_OUTPUT_PATH, index=False)

print(f"🎯 ĐÃ XUẤT FILE SUBMISSION THÀNH CÔNG!")
print(f"💾 File submission (.csv) đã được lưu tại: {SUBMISSION_OUTPUT_PATH}")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.1 MB/s eta 0:00:00
🚀 Khởi tạo Pipeline Đánh Giá OCR & Tạo Submission. Thiết bị sử dụng: CUDA
🔄 Đang nạp TrOCR Processor từ: microsoft/trocr-base-handwritten
🔄 Đang nạp TrOCR Model từ: Kansallisarkisto/cyrillic-htr-model...


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

✨ Thêm các ký tự đặc trưng của tiếng Ukraine vào Vocab...
🔹 Đã thêm thành công 9 token mới vào Tokenizer.
🔹 Đã cập nhật kích thước Embedding của Decoder thành: 50274
✅ Khởi tạo mô hình và mở rộng Vocab thành công!
⚡ Bắt đầu chạy inference trên 200 ảnh từ tập test...



Evaluating: 100%|██████████| 200/200 [1:46:48<00:00, 32.04s/it]



🔄 Đang tính toán các chỉ số lỗi (Metrics Calculation)...

🏁 TIẾN TRÌNH ĐÁNH GIÁ HOÀN TẤT!
💾 File báo cáo metric đã được lưu tại: /kaggle/working/report.json
                 CER      WER  Exact_Match  Total_Samples
handwritten  0.88090  1.04563      0.00445         3149.0
printed      0.91329  1.15282      0.00000           68.0
annotation   0.66903  0.95690      0.22078           77.0
overall      0.88062  1.04696      0.00941         3294.0

📦 ĐANG XUẤT FILE SUBMISSION CSV...
🎯 ĐÃ XUẤT FILE SUBMISSION THÀNH CÔNG!
💾 File submission (.csv) đã được lưu tại: /kaggle/working/submission.csv
